# <center>Data Wrangling</center>

<img src="../image/wrangling_DALLE.jpeg" width=30% align="right" style="in-line">

>*80% of data wrangling is spent cleaning the data.*
>
>*The other 20% is spent complaining about cleaning the data.*
>
>— Source: [Kyle Bradbury & Nick Eubank](https://www.practicaldatascience.org/notebooks/class_3/week_3/20_cleaning_identifying.html)


## Learning goals

1. Scrape across multiple HTML pages.
2. Load and identify the structure of datasets using Python.
3. Become familiar with common data wrangling tasks, including handling missing values, merging datasets, filtering, and sorting data.

## Agenda

1. [Review: web scraping](#1)
2. [Data wrangling](#2)

<a name="1"></a>
## Agenda 1. Web scraping

&#x1F4DD; **<font color=dodgerblue>FROM LAST WEEK: </font> We would like to find out what the European Union has done recently to advance sustainable mobility and transport. One possible data source is the news we just scraped, but we need more information than just the titles.**

Your task is to collect the news items published **from 2025 onward**. You can choose how far you want to go with this task.

1. **Starting point:** Print out **the publication date, the title, the short description, the news type, and the link to the full article** for a few news items.
2. **Next step:** Once that works, organize the information from the first page and save it as a **csv** file. The file should have one row per news item and contain the date, title, short description, news type, and link.
3. **Further challenge:** Collect all news items published from 2025 onward and save them in a csv file with the same columns.

Choose the amount of challenge you want. This task will help you practice collecting information from the web, organizing scraped data, and saving it in a format that can be reused for analysis. Try not to ask AI to write the entire code for you.

Here are some tips:
1. How many pages do you need to scrape? Observe how the web address changes between the first page and the second.
2. Remember that we talked about **avoiding overloading servers** in the ethics part. Make sure to use `time.sleep()`.

In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import csv
import pandas as pd

### Step 1: Send an HTTP request to retrieve the web page content

In [ ]:
url = "https://transport.ec.europa.eu/news-events/news_en?page=0"
page = requests.get(url)

In [ ]:
print(page)

### Step 2: Parse the HTML content

In [ ]:
soup = BeautifulSoup(page.content, 'html.parser')

In [ ]:
print(soup.prettify())

### Step 3: Locate the HTML elements containing the desired data

The website: https://transport.ec.europa.eu/news-events/news_en?page=0

Chrome: Right click -> `Inspect`

In [ ]:
news = soup.find_all("article", class_="ecl-content-item")

In [ ]:
#print(news)

In [ ]:
len(news)

### Step 4: Extract data

In [ ]:
for item in news:
    date = item.time.attrs['datetime']
    title = item.find("a", class_="ecl-link--standalone").get_text(strip=True)
    desc = item.find("div", class_="ecl-content-block__description").get_text(strip=True)
    news_type = item.find("li", class_="ecl-content-block__primary-meta-item").get_text(strip=True)
    link = item.find("a", class_="ecl-link--standalone", href=True)["href"]
    print(date, title, desc)
    print(news_type)
    print(link)
    print("=====")

In [ ]:
news[0].time # the <time> element of the first article

### Step 5: Save data to csv

To scrape multiple pages, we need to observe how the web addresses change.

In [ ]:
#page_number = range(10,27) # year 2025. your number could be different
#page_number = range(0,12) # year 2026
stem = "https://transport.ec.europa.eu/news-events/news_en?page="

In [ ]:
# give the csv file a name
#csv_filename = "news2025.csv" 
#csv_filename = "news2026.csv"

In [ ]:
with open(csv_filename, "w", newline="", encoding="utf-8") as f: # open the csv file 
    writer = csv.writer(f)
    writer.writerow(["date", "title", "desc", "news_type", "link"]) # header row

    for i in page_number: # loop over all the pages
        url = stem + str(i)
        print(url)

        page = requests.get(url) # send the request
        soup = BeautifulSoup(page.content, 'html.parser') # parse the content
        news = soup.find_all("article", class_="ecl-content-item") # locate all the news

        for item in news: # loop over each article
            try: # use try except to skip errors, e.g., NoneType object - empty elements
                date = item.time.attrs['datetime']
            except (AttributeError, KeyError, TypeError) as e:
                date = ""
                print(f"Error extracting date: {e}")
                print(item)
                print("====")
            try:
                title = item.find("a", class_="ecl-link--standalone").get_text(strip=True)
            except AttributeError as e:
                title = ""
                print(f"Error extracting title: {e}")
                print(item)
                print("====")
            try:
                desc = item.find("div", class_="ecl-content-block__description").get_text(strip=True)
            except AttributeError as e:
                desc = ""
                print(f"Error extracting description: {e}")
                print(item)
                print("====")
            try:
                news_type = item.find("li", class_="ecl-content-block__primary-meta-item").get_text(strip=True)
            except AttributeError as e:
                news_type = ""
                print(f"Error extracting news type: {e}")
                print(item)
                print("====")
            try:
                link = item.find("a", class_="ecl-link--standalone", href=True)["href"]
            except (AttributeError, TypeError) as e:
                link = ""
                print(f"Error extracting link: {e}")
                print(item)
                print("====")
            writer.writerow([date, title, desc, news_type, link])

        time.sleep(2) # be polite to the server

<a name="2"></a>
## Agenda 2. Data wrangling

Data wrangling is the process of converting raw data into a usable form. It typically involves examining the data, handling missing values, cleaning, and transforming it.

### Examining data

We start by loading our datasets and inspecting them to get a better sense of their structure. 

In [ ]:
news2026 = pd.read_csv("news2026.csv", encoding="utf-8", header=0, names=["date", "title", "desc", "type", "link"]) # import csv with column names
news2025 = pd.read_csv("news2025.csv", encoding="utf-8", header=0, names=["date", "title", "desc", "type", "link"])

In [ ]:
news2026.head()

In [ ]:
len(news2026) # check the number of rows

In [ ]:
news2026.shape # check the number of rows and columns

In [ ]:
news2026.info() # summarize by the columns

In [ ]:
# news2026.describe() # useful for numerical data

In [ ]:
type(news2026.date.iloc[0]) # check the data type

In [ ]:
news2026.type.unique() # check the unique values of a column

In [ ]:
news2026.type.value_counts() # check each value's number of occurrences

### Handling missing values

In [ ]:
news2026.isnull().sum() # identify missing data

In [ ]:
news2025.isnull().sum()

In [ ]:
news2026[news2026.title.isnull()] # pinpoint which rows have missing values

Missing values can often skew analyses or create errors during analysis. Here are some common strategies for handling missing values:

- Manually review and fix the missing data (if possible)
- Remove missing data e.g., drop rows
- Impute missing data e.g., replace with the mean (for numerical data)
- Fill with a specific value e.g., replace with "unknown" or "NA"
- Predictive imputation e.g., use machine learning models to predict missing values based on other features
- Leave as missing

By the looks of it, our example has no missing values.

In [ ]:
news2026[news2026.title == news2026.desc] # maybe a hidden problem

### Merging datasets

There are many different ways of merging datasets. Check this [documentation](https://pandas.pydata.org/docs/user_guide/merging.html). Let's combine our two datasets into one.

In [ ]:
news = pd.concat([news2025, news2026], ignore_index=True) # concatenate and re-index

In [ ]:
news

In [ ]:
news.isnull().sum()

Nothing is missing. However, we might have created duplicated values when merging datasets.

### Cleaning

In [ ]:
news[news.duplicated()] # identify duplicated rows

In [ ]:
news.duplicated() # returns a Boolean Series indicating whether each row is a duplicate of a previous row

In [ ]:
news = news.drop_duplicates() # drop duplicated rows

In [ ]:
news

We also want to fix the date using [`pandas.to_datetime` function](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html). 

In [ ]:
type(news.date[0])

In [ ]:
news.loc[:, "date"] = pd.to_datetime(news.date) # change the data type to date

In [ ]:
type(news.date[0])

In [ ]:
news

In [ ]:
news = news.sort_values(by="date", ascending=False) # sort the rows by date

In [ ]:
news

In [ ]:
news['date'] = pd.to_datetime(news['date']) # make sure the 'date' column is of datetime type
news = news[news['date'].dt.year!=2024] # remove news from 2024

In [ ]:
news

In [ ]:
news.type.unique()

In [ ]:
news[news.type=="Speech"]

In [ ]:
news[news.type=="General publications"]

Let's only keep the news.

In [ ]:
news = news[news.type=="News article"]

In [ ]:
news = news.reset_index(drop=True) # reset the index

In [ ]:
news

In [ ]:
news = news[["date", "title", "desc"]] # new df with selected columns (if needed)

In [ ]:
news.to_csv("news_clean.csv", index=False) # save to csv

&#x2753;**<font color=cornflowerblue>QUESTION: </font> What are the common tasks involved in data wrangling?**

&#x270A; **<font color=firebrick>DO THIS: </font> Here are two options to practice what we've learned and to further develop your data skills:**

1. Do some data wrangling on the data you scraped last week.
2. Explore the existing `news_clean.csv` dataset. What research question can it answer? Using what methods/tools?

Feel free to pick one task or do both. Have fun!

In [ ]:
# put your code here













---------
### Congratulations, we are done!

This notebook is written by [Meng Cai](https://www.verkehr.tu-darmstadt.de/vv/das_institut_ivv/team_ivv/wissenschaftliche_mitarbeiter_doktoranden/meng_cai/standardseite_204.de.jsp), Technical University of Darmstadt. This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.
<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a>